In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder


df = pd.read_csv("hotel_bookings.csv")

print("Head: ")
print(df.head())
print("\nShape: ")
print(df.shape)
print("\nInfo: ")
print(df.info())
print("\nDescribe: ")
print(df.describe().T)
print("\nDTypes: ")
print(df.dtypes)

print("\n Target Class Distribution (is_canceled): ")
print(df["is_canceled"].value_counts(normalize=True) * 100)

y = df["is_canceled"]
X = df.drop(columns=["is_canceled"])



Head: 
          hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  \
0  Resort Hotel            0        342               2015               July   
1  Resort Hotel            0        737               2015               July   
2  Resort Hotel            0          7               2015               July   
3  Resort Hotel            0         13               2015               July   
4  Resort Hotel            0         14               2015               July   

   arrival_date_week_number  arrival_date_day_of_month  \
0                        27                          1   
1                        27                          1   
2                        27                          1   
3                        27                          1   
4                        27                          1   

   stays_in_weekend_nights  stays_in_week_nights  adults  ...  deposit_type  \
0                        0                     0       2  ...    No Deposit   

#### Task 2 - Missing Values, Leakage, and Outliers

In [15]:
missing_count = X.isnull().sum()
missing_percent = (missing_count / len(X)) * 100
missing_df = pd.DataFrame({'Missing Count' : missing_count, 'Missing_Percentage (%)' : missing_percent})
print("\nMissing Value Analysis:")
print(missing_df[missing_df['Missing Count'] > 0])

cols_to_drop = ['company', 'reservation_status', 'reservation_status_date']
cols_to_drop = [col for col in cols_to_drop if col in X.columns]

X = X.drop(columns=cols_to_drop)
print(f"\nDropped columns due to high missingness or targer leakage: {cols_to_drop}")

initial_row_count = len(X)

valid_rows = (
    (X['adr'] >= 0) & (X['adr'] < 5000) &
    (X['adults'] > 0) &
    (X['adults'] <= 10)
)

X = X[valid_rows]
y = y[valid_rows]

rows_removed = initial_row_count - len(X)
print(f"Rows removed fue to extreme outliers: {rows_removed} ({rows_removed / initial_row_count * 100:.2f}%) ")

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"\nNumerical Columns ({len(num_cols)}): {num_cols}")
print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}")


Missing Value Analysis:
          Missing Count  Missing_Percentage (%)
children              4                0.003350
country             488                0.408744
agent             16340               13.686238
company          112593               94.306893

Dropped columns due to high missingness or targer leakage: ['company', 'reservation_status', 'reservation_status_date']
Rows removed fue to extreme outliers: 417 (0.35%) 

Numerical Columns (18): ['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'agent', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']
Categorical Columns (10): ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 

C:\Users\choks\AppData\Local\Temp\ipykernel_26660\630262594.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()


#### Task 3 - Create Two Preprocessing Pipelines

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


